# 42 · Analysis: graduates vs openings, and the crowded-field grid

**Rules:** cutoffs fixed before this notebook ran (growing = 2022-32 change > 0; many = more than 1.0 graduate per opening; crowded = shrinking and many, only; words: "more competition", never "bad major"). Reads: `fct_degree_awards` (IPEDS, all NYC colleges), `dim_college`, `dim_major`, `dim_major_award`, `bridge_major_job` (D2/D16: degree-level filter, openings-weighted split, 15% filter), `fct_job_projection` 2022-32, `dim_job`. Re-run under D20: majors whose ACS field holds the bridge's jobs at the population rate (`dim_major_award.link_status = dispersed`, from nb 44's lift test) are no longer counted as supply; the D16-as-built numbers are kept as a sensitivity row. Changes nothing.

**Questions**
1. How many graduates finish each major a year, is that rising, and how much of it is CUNY?
2. At the job: graduates a year flowing to the job (D2 split) against the job's NYC openings a year. S11 (1).
3. At the major: what a graduate faces, averaged over the jobs the major leads to. S11 (2).
4. How much do the answers move under other reasonable cutoffs and weights? S11 (3).

Limits stated with every number (S11 5): NYC employers hire from everywhere and graduates leave; openings are a ten-year average and include experienced hires; the ratio compares two flows, it is not a placement rate; the split of a major across jobs is a model (D2), shown as a range.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from explore import q

pd.set_option("display.max_columns", 40, "display.max_colwidth", 50, "display.width", 240,
              "display.float_format", "{:,.2f}".format)
LEVEL = {3: "associate's", 5: "bachelor's"}
GROW_CUT, RATIO_CUT = 0.0, 1.0   # plan §3c


def md(s):
    display(Markdown(s))


def show(df, title=None):
    if title:
        md(f"**{title}**")
    display(df)


def box(growth, ratio, gcut=GROW_CUT, rcut=RATIO_CUT):
    g = np.where(growth > gcut, "growing", "shrinking")
    m = np.where(ratio > rcut, "many grads", "few grads")
    return pd.Series([f"{a} · {b}" for a, b in zip(g, m)], index=growth.index)

## 1. Supply: graduates by major, 2015 → 2024
First majors only (`major_number = 1`, nb 05 rule), associate's and bachelor's, every NYC college; CUNY = `dim_college.is_cuny`. 2024 is the latest IPEDS year (provisional).

In [2]:
aw = q("""
select a.year, a.cip_code, a.award_level, a.unitid, c.is_cuny, a.awards_total
from fct_degree_awards a join dim_college c using (unitid)
where a.major_number = 1 and a.award_level in (3, 5)
""")
maj = q("select cip_code, cip_title_clean as major, cip_family_title_clean as family from dim_major").set_index("cip_code")
print(f"fct_degree_awards first-major rows, levels 3 and 5: {len(aw):,}; awards 2024 = {aw[aw.year == 2024].awards_total.sum():,.0f}")
yr = aw.pivot_table(index="year", columns=["award_level", "is_cuny"], values="awards_total", aggfunc="sum")
yr.columns = [f"{LEVEL[l]} · {'CUNY' if c else 'other NYC'}" for l, c in yr.columns]
yr["associate's · all"] = yr.filter(like="associate").sum(axis=1)
yr["bachelor's · all"] = yr.filter(like="bachelor").sum(axis=1)
show(yr.round(0).astype(int), "NYC first-major awards by year (IPEDS)")
show(((yr.loc[2024] / yr.loc[2019] - 1) * 100).round(1).rename("2019 → 2024 %").to_frame().T, "Growth 2019 → 2024")

fct_degree_awards first-major rows, levels 3 and 5: 23,214; awards 2024 = 73,225


**NYC first-major awards by year (IPEDS)**

,associate's · other NYC,associate's · CUNY,bachelor's · other NYC,bachelor's · CUNY,associate's · all,bachelor's · all
year,,,,,,
2015,10734,15478,29886,22736,26212,52622
2016,10107,16345,30475,23709,26452,54184
2017,9139,17372,29799,24456,26511,54255
2018,8791,18247,29785,25371,27038,55156
2019,8713,18880,30509,27273,27593,57782
2020,8319,18498,32114,27219,26817,59333
2021,8349,17596,31295,29590,25945,60885
2022,7554,14662,31749,28008,22216,59757
2023,6717,12246,30949,25074,18963,56023


**Growth 2019 → 2024**

,associate's · other NYC,associate's · CUNY,bachelor's · other NYC,bachelor's · CUNY,associate's · all,bachelor's · all
2019 → 2024 %,-18.70,-39.60,2.60,-14.10,-33.00,-5.30


In [3]:
A = aw.groupby(["cip_code", "award_level", "year"]).awards_total.sum().unstack("year").fillna(0)
cuny24 = aw[(aw.year == 2024) & aw.is_cuny].groupby(["cip_code", "award_level"]).awards_total.sum()
top = pd.DataFrame({"awards 2024": A[2024], "awards 2019": A[2019], "growth %": (A[2024] / A[2019].replace(0, np.nan) - 1) * 100,
                    "CUNY 2024": cuny24, "CUNY share %": cuny24 / A[2024] * 100}).reset_index()
top["major"] = top.cip_code.map(maj.major)
for lvl in (5, 3):
    show(top[top.award_level == lvl].nlargest(15, "awards 2024").set_index(["cip_code", "major"]).drop(columns="award_level").round(1),
         f"Largest {LEVEL[lvl]} majors, NYC 2024")

**Largest bachelor's majors, NYC 2024**

,,awards 2024,awards 2019,growth %,CUNY 2024,CUNY share %
cip_code,major,,,,,
42.0101,"Psychology, General","4,129.00","4,375.00",-5.60,"2,927.00",70.90
51.3801,Registered Nursing/Registered Nurse,"2,197.00","2,127.00",3.30,576.00,26.20
52.0201,"Business Administration and Management, General","1,922.00","2,402.00",-20.00,"1,032.00",53.70
26.0101,"Biology/Biological Sciences, General","1,715.00","1,664.00",3.10,865.00,50.40
52.0301,Accounting,"1,630.00","2,557.00",-36.30,"1,239.00",76.00
43.0103,Criminal Justice/Law Enforcement Administration,"1,607.00","1,904.00",-15.60,"1,264.00",78.70
11.0701,Computer Science,"1,465.00",947.00,54.70,874.00,59.70
52.0801,"Finance, General","1,299.00","1,758.00",-26.10,972.00,74.80
45.1001,"Political Science and Government, General","1,276.00","1,380.00",-7.50,498.00,39.00


**Largest associate's majors, NYC 2024**

,,awards 2024,awards 2019,growth %,CUNY 2024,CUNY share %
cip_code,major,,,,,
24.0101,Liberal Arts and Sciences/Liberal Studies,"2,466.00","6,184.00",-60.10,"2,210.00",89.60
51.3801,Registered Nursing/Registered Nurse,"1,539.00","1,249.00",23.20,730.00,47.40
52.0201,"Business Administration and Management, General","1,451.00","2,807.00",-48.30,"1,216.00",83.80
52.1902,Fashion Merchandising,736.00,832.00,-11.50,33.00,4.50
38.0206,Jewish/Judaic Studies,519.00,551.00,-5.80,NaN,NaN
51.0801,Medical/Clinical Assistant,494.00,833.00,-40.70,48.00,9.70
42.0101,"Psychology, General",473.00,402.00,17.70,473.00,100.00
51.0000,"Health Services/Allied Health/Health Sciences, General",449.00,364.00,23.40,410.00,91.30
43.0103,Criminal Justice/Law Enforcement Administration,440.00,"1,633.00",-73.10,436.00,99.10


### How much of the supply has a job link?
`dim_major_award.link_status`: `linked` = at least one job passes the D2 degree-level filter at 15% and the ACS test (nb 44, D20) does not reject the link; `dispersed` (bachelor's only, D20, re-run) = a job passes, but the major's ACS field of degree holds the bridge's jobs at the population rate (lift < 1.5: Psychology, Political Science, Criminal Justice, Business Management and Administration, General Business), so its graduates are not counted as supply to any job in this notebook; `transfer_pathway` = the major's jobs need a higher degree (most associate's programs, D14); `not_in_crosswalk` = no NCES CIP–SOC link at all.

In [4]:
dma = q("select major_award_key, cip_code, award_level, link_status, n_pass_15, n_all_other_jobs, field_fod1p, field_lift, field_verdict from dim_major_award")
a24 = aw[aw.year == 2024].groupby(["cip_code", "award_level"]).awards_total.sum().rename("awards").reset_index().merge(dma, on=["cip_code", "award_level"], how="left")
assert a24.link_status.notna().all()
cov = a24.pivot_table(index="award_level", columns="link_status", values="awards", aggfunc="sum").fillna(0)
cov["share linked %"] = cov["linked"] / cov.sum(axis=1) * 100
cov["share dispersed %"] = cov.get("dispersed", 0) / cov.sum(axis=1) * 100
cov.index = cov.index.map(LEVEL)
show(cov.round(1), "2024 awards by link status")
DISPERSED = set(dma[dma.link_status == "dispersed"].major_award_key)
disp = a24[a24.link_status == "dispersed"].sort_values("awards", ascending=False)
print(f"dispersed majors (D20): {len(DISPERSED)} in the dim, {int((disp.awards > 0).sum())} with 2024 awards, {disp.awards.sum():,.0f} awards")
show(disp.head(12).assign(major=lambda d: d.cip_code.map(maj.major))[["cip_code", "major", "awards", "field_fod1p", "field_lift", "n_pass_15", "n_all_other_jobs"]].round(2).reset_index(drop=True),
     "The largest dispersed majors (ACS lift of their field under 1.5; excluded from supply below)")

**2024 awards by link status**

link_status,dispersed,linked,not_in_crosswalk,transfer_pathway,share linked %,share dispersed %
award_level,,,,,,
associate's,0.00,"6,108.00",513.00,"11,869.00",33.00,0.00
bachelor's,"12,067.00","37,380.00","1,575.00","3,713.00",68.30,22.00


dispersed majors (D20): 73 in the dim, 22 with 2024 awards, 12,067 awards


**The largest dispersed majors (ACS lift of their field under 1.5; excluded from supply below)**

,cip_code,major,awards,field_fod1p,field_lift,n_pass_15,n_all_other_jobs
0,42.0101,"Psychology, General",4129,5200,1.03,2,1
1,52.0201,"Business Administration and Management, General",1922,6203,1.36,22,3
2,43.0103,Criminal Justice/Law Enforcement Administration,1607,5301,1.40,2,1
3,45.1001,"Political Science and Government, General",1276,5506,1.19,4,1
4,42.2799,"Research and Experimental Psychology, Other",853,5200,1.03,1,1
5,52.1301,Management Science,821,6203,1.36,5,0
6,42.2812,Forensic Psychology,476,5200,1.03,1,1
7,52.0101,"Business/Commerce, General",415,6200,1.19,15,2
8,52.0701,Entrepreneurship/Entrepreneurial Studies,103,6203,1.36,3,1
9,52.0213,Organizational Leadership,80,6203,1.36,5,1


## 2. The job grid (S11 1): graduates per opening, by job
**Supply to a job** = Σ over linked majors of (2024 awards × the major's openings weight for that job). Openings = 2022-32 annual average for the job (D1 fill rows flagged). Jobs whose openings NYS DOL does not publish and D13 could not estimate (`openings_basis = none`) get no ratio.

In [5]:
bridge = q("""
select major_award_key, cip_code, award_level, soc_code, weight_openings, weight_equal, openings_nyc, openings_basis, is_all_other, n_jobs_in_major
from bridge_major_job
""")
proj = q("select soc_code, pct_change, annual_openings_avg, quality_flag from fct_job_projection where edition = '2022-2032' and is_leaf")
jobs = q("select soc_code, soc_title, entry_education, is_degree_entry, is_all_other, in_tech, in_health, in_business, in_education from dim_job")
print(f"bridge rows: {len(bridge):,} ({bridge.major_award_key.nunique()} major × level keys); no-openings links: {int((bridge.openings_basis == 'none').sum())}")
wsum = bridge.groupby("major_award_key").weight_openings.sum(min_count=1)
nullw = wsum[wsum.isna()]
print(f"majors whose every passing job has no published or estimated openings (weights null, no ratio possible): {len(nullw)}")
assert (wsum.dropna().sub(1).abs() < 1e-6).all(), "openings weights must sum to 1 per major"
bridge["weight_openings"] = bridge.weight_openings.fillna(0.0)
bridge["weight_equal"] = bridge.weight_equal.fillna(0.0)

B_all = bridge.merge(a24[["cip_code", "award_level", "awards"]], on=["cip_code", "award_level"], how="left")
B_all["awards"] = B_all.awards.fillna(0)   # majors with no 2024 awards contribute nothing
# D20: dispersed majors keep their bridge rows but are not supply to a job. B_all (D16 A as built) is kept for the sensitivity table.
B = B_all[~B_all.major_award_key.isin(DISPERSED)].copy()
print(f"bridge rows after D20: {len(B):,} of {len(B_all):,} (dispersed rows removed: {len(B_all) - len(B):,}, carrying {B_all[B_all.major_award_key.isin(DISPERSED)].drop_duplicates('major_award_key').awards.sum():,.0f} awards)")


def job_table(B, wcol="weight_openings", awcol="awards"):
    B = B.copy()
    B["supply"] = B[awcol] * B[wcol]
    J = B.groupby("soc_code").agg(supply=("supply", "sum"), n_majors=("cip_code", "nunique"),
                                   supply_assoc=("supply", lambda s: s[B.loc[s.index, "award_level"] == 3].sum()))
    J = J.join(proj.set_index("soc_code")[["pct_change", "annual_openings_avg", "quality_flag"]], how="left")
    J = J.join(jobs.set_index("soc_code")[["soc_title", "entry_education", "is_all_other"]])
    J["ratio"] = J.supply / J.annual_openings_avg
    J["box"] = box(J["pct_change"].fillna(0), J.ratio.fillna(0))
    J.loc[J.ratio.isna(), "box"] = "no openings published"
    return J


J = job_table(B)
print(f"jobs receiving graduates: {len(J)}; with a ratio: {int(J.ratio.notna().sum())}; Σ supply = {J.supply.sum():,.0f} "
      f"(= 2024 awards at linked majors {a24[a24.link_status == 'linked'].awards.sum():,.0f}, minus weight on no-openings jobs)")
grid = J.groupby("box").agg(jobs=("soc_title", "size"), openings=("annual_openings_avg", "sum"), graduates=("supply", "sum")).round(0)
show(grid, "Job grid: counts per box (cutoffs: growth > 0, ratio > 1.0)")
cols = ["soc_title", "entry_education", "supply", "annual_openings_avg", "ratio", "pct_change", "n_majors", "is_all_other", "quality_flag"]
show(J[J.box == "shrinking · many grads"].sort_values("supply", ascending=False)[cols].round(2), "Shrinking · many graduates per opening (\"more competition\")")
show(J[J.box == "growing · many grads"].sort_values("ratio", ascending=False)[cols].head(20).round(2), "Growing · many graduates per opening (top 20 by ratio)")
show(J[J.ratio.notna()].nlargest(20, "supply")[cols].round(2), "Jobs receiving the most graduates a year")

bridge rows: 3,802 (1860 major × level keys); no-openings links: 116
majors whose every passing job has no published or estimated openings (weights null, no ratio possible): 28
bridge rows after D20: 3,559 of 3,802 (dispersed rows removed: 243, carrying 12,067 awards)


jobs receiving graduates: 427; with a ratio: 333; Σ supply = 43,488 (= 2024 awards at linked majors 43,488, minus weight on no-openings jobs)


**Job grid: counts per box (cutoffs: growth > 0, ratio > 1.0)**

,jobs,openings,graduates
box,,,
growing · few grads,265,"226,165.00","25,507.00"
growing · many grads,21,"7,389.00","13,758.00"
no openings published,94,0.00,943.00
shrinking · few grads,46,"36,925.00","3,212.00"
shrinking · many grads,1,49.00,67.00


**Shrinking · many graduates per opening ("more competition")**

,soc_title,entry_education,supply,annual_openings_avg,ratio,pct_change,n_majors,is_all_other,quality_flag
soc_code,,,,,,,,,
17-3013,Mechanical Drafters,Associate's degree,67.17,49.00,1.37,-1.56,7,False,published


**Growing · many graduates per opening (top 20 by ratio)**

,soc_title,entry_education,supply,annual_openings_avg,ratio,pct_change,n_majors,is_all_other,quality_flag
soc_code,,,,,,,,,
17-3029,"Engineering Technologists and Technicians, Exc...",Associate's degree,15.29,1.01,15.17,7.25,34,True,imputed
11-9121,Natural Sciences Managers,Bachelor's degree,631.95,96.00,6.58,16.84,146,False,published
29-1128,Exercise Physiologists,Bachelor's degree,136.00,25.00,5.44,16.67,3,False,published
19-1029,"Biological Scientists, All Other",Bachelor's degree,213.28,62.00,3.44,19.05,63,True,published
11-9041,Architectural and Engineering Managers,Bachelor's degree,618.78,197.00,3.14,14.98,69,False,published
11-9199,"Managers, All Other",Bachelor's degree,"2,693.50",858.84,3.14,1.41,92,True,imputed
27-1027,Set and Exhibit Designers,Bachelor's degree,277.24,126.00,2.20,3.90,3,False,published
27-1029,"Designers, All Other",Bachelor's degree,170.45,82.00,2.08,2.22,3,True,published
19-4021,Biological Technicians,Bachelor's degree,256.93,125.00,2.06,17.44,19,False,published


**Jobs receiving the most graduates a year**

,soc_title,entry_education,supply,annual_openings_avg,ratio,pct_change,n_majors,is_all_other,quality_flag
soc_code,,,,,,,,,
25-2031,"Secondary School Teachers, Except Special and ...",Bachelor's degree,"4,506.77","2,627.00",1.72,18.43,88,False,published
29-1141,Registered Nurses,Bachelor's degree,"3,830.00","7,410.00",0.52,24.90,21,False,published
11-9199,"Managers, All Other",Bachelor's degree,"2,693.50",858.84,3.14,1.41,92,True,imputed
27-2012,Producers and Directors,Bachelor's degree,"1,492.84","2,542.00",0.59,1.72,10,False,published
15-1252,Software Developers,Bachelor's degree,"1,346.17","4,308.19",0.31,11.25,20,False,imputed
13-2011,Accountants and Auditors,Bachelor's degree,"1,145.66","6,648.00",0.17,7.97,6,False,published
41-4012,"Sales Representatives, Wholesale and Manufactu...",High school diploma or equivalent,"1,085.85","2,604.00",0.42,-5.99,6,False,published
15-2051,Data Scientists,Bachelor's degree,"1,072.46",811.38,1.32,33.33,34,False,imputed
11-3021,Computer and Information Systems Managers,Bachelor's degree,948.07,"1,848.00",0.51,14.19,20,False,published


In [6]:
nosupply = jobs[jobs.is_degree_entry & ~jobs.soc_code.isin(J.index)].merge(proj, on="soc_code")
show(nosupply.nlargest(15, "annual_openings_avg")[["soc_code", "soc_title", "entry_education", "annual_openings_avg", "pct_change"]].round(1),
     f"Degree-reachable jobs with openings but no linked NYC major ({len(nosupply)} jobs, {nosupply.annual_openings_avg.sum():,.0f} openings a year)")

**Degree-reachable jobs with openings but no linked NYC major (8 jobs, 6,028 openings a year)**

,soc_code,soc_title,entry_education,annual_openings_avg,pct_change
5,25-3031,"Substitute Teachers, Short-Term",Bachelor's degree,"2,370.00",22.20
1,13-1020,Buyers and Purchasing Agents,Bachelor's degree,"1,441.00",11.10
6,29-2010,Clinical Laboratory Technologists and Technicians,Bachelor's degree,808.00,24.70
4,25-2052,"Special education teachers, kindergarten and e...",Bachelor's degree,703.00,18.70
2,13-1081,Logisticians,Bachelor's degree,225.00,22.10
0,11-3061,Purchasing Managers,Bachelor's degree,172.00,11.30
7,41-9031,Sales Engineers,Bachelor's degree,158.00,4.80
3,13-2020,Property Appraisers and Assessors,Bachelor's degree,151.00,6.60


## 3. The major grid (S11 2): what a graduate of each major faces
For each major × level: ratio = Σ_j weight_j × ratio_j and growth = Σ_j weight_j × change_j over the jobs with a ratio (weights renormalized; `w_no_ratio` says how much weight had no openings).

In [7]:
def major_table(B, J, wcol="weight_openings"):
    M = B.merge(J[["ratio", "pct_change"]], left_on="soc_code", right_index=True, how="left")
    M["w"] = M[wcol]
    ok = M[M.ratio.notna()].copy()
    ok["w_r"] = ok.w / ok.groupby("major_award_key").w.transform("sum")
    out = ok.groupby("major_award_key").apply(lambda d: pd.Series({
        "ratio": (d.w_r * d.ratio).sum(), "growth": (d.w_r * d["pct_change"]).sum(), "n_jobs": len(d),
        "top job": d.sort_values("w", ascending=False).soc_code.iloc[0], "top job weight": d.w.max()}), include_groups=False)
    keys = B.drop_duplicates("major_award_key").set_index("major_award_key")[["cip_code", "award_level", "awards"]]
    out = out.join(keys)
    out["w_no_ratio"] = 1 - M.groupby("major_award_key").apply(lambda d: d.loc[d.ratio.notna(), "w"].sum(), include_groups=False)
    out["major"] = out.cip_code.map(maj.major)
    out["level"] = out.award_level.map(LEVEL)
    out["box"] = box(out.growth, out.ratio)
    return out


Mj = major_table(B, J)
Mj["top job"] = Mj["top job"].map(jobs.set_index("soc_code").soc_title)
print(f"majors × level with a score: {len(Mj)}; 2024 awards covered: {Mj.awards.sum():,.0f}")
mg = Mj.groupby(["level", "box"]).agg(majors=("major", "size"), awards_2024=("awards", "sum")).round(0)
show(mg, "Major grid: counts per box")
mcols = ["level", "major", "awards", "ratio", "growth", "n_jobs", "top job", "top job weight", "w_no_ratio"]
show(Mj[(Mj.box == "shrinking · many grads") & (Mj.awards > 0)].sort_values("awards", ascending=False)[mcols].round(2),
     "Majors where graduates face more competition (shrinking jobs, more than one graduate per opening), by 2024 awards")
show(Mj.nlargest(25, "awards")[mcols + ["box"]].round(2), "The 25 largest majors and their box")

majors × level with a score: 1614; 2024 awards covered: 43,351


**Major grid: counts per box**

majors  awards_2024
level       box                                        
associate's growing · few grads        204     4,662.00
            growing · many grads        48       645.00
            shrinking · few grads       16       628.00
            shrinking · many grads       4        72.00
bachelor's  growing · few grads        858    21,131.00
            growing · many grads       421    14,136.00
            shrinking · few grads       59     1,990.00
            shrinking · many grads       4        87.00

**Majors where graduates face more competition (shrinking jobs, more than one graduate per opening), by 2024 awards**

,level,major,awards,ratio,growth,n_jobs,top job,top job weight,w_no_ratio
major_award_key,,,,,,,,,
15.0805|5,bachelor's,Mechanical/Mechanical Engineering Technology/T...,87.00,1.37,-1.56,1,Mechanical Drafters,0.41,0.59
15.0805|3,associate's,Mechanical/Mechanical Engineering Technology/T...,68.00,1.37,-1.56,1,Mechanical Drafters,0.41,0.59
15.1306|3,associate's,Mechanical Drafting and Mechanical Drafting CA...,4.00,1.37,-1.56,1,Mechanical Drafters,1.00,0.00


**The 25 largest majors and their box**

,level,major,awards,ratio,growth,n_jobs,top job,top job weight,w_no_ratio,box
major_award_key,,,,,,,,,,
51.3801|5,bachelor's,Registered Nursing/Registered Nurse,"2,197.00",0.52,24.90,1,Registered Nurses,1.00,0.00,growing · few grads
26.0101|5,bachelor's,"Biology/Biological Sciences, General","1,715.00",1.93,18.35,4,"Secondary School Teachers, Except Special and ...",0.84,0.07,growing · many grads
52.0301|5,bachelor's,Accounting,"1,630.00",0.19,9.22,7,Accountants and Auditors,0.69,0.00,growing · few grads
51.3801|3,associate's,Registered Nursing/Registered Nurse,"1,539.00",0.52,24.90,1,Registered Nurses,1.00,0.00,growing · few grads
11.0701|5,bachelor's,Computer Science,"1,465.00",0.53,13.18,12,Software Developers,0.43,0.00,growing · few grads
52.0801|5,bachelor's,"Finance, General","1,299.00",0.10,8.93,10,General and Operations Managers,0.48,0.02,growing · few grads
45.0603|5,bachelor's,Econometrics and Quantitative Economics,"1,014.00",2.06,15.84,5,"Managers, All Other",0.41,0.00,growing · many grads
11.0101|5,bachelor's,"Computer and Information Sciences, General",916.00,0.49,10.21,10,Computer and Information Systems Managers,0.32,0.00,growing · few grads
23.0101|5,bachelor's,"English Language and Literature, General",837.00,1.65,17.75,2,"Secondary School Teachers, Except Special and ...",0.95,0.00,growing · many grads


## 4. Sensitivity (S11 3): how much the "more competition" box moves
Each row changes one thing. "Crowded awards" = 2024 awards in majors that land in shrinking · many under that variant.

In [8]:
def crowded(Mv, gcut=GROW_CUT, rcut=RATIO_CUT):
    b = box(Mv.growth, Mv.ratio, gcut, rcut)
    sel = Mv[b == "shrinking · many grads"]
    return len(sel), sel.awards.sum(), len(Mv[(b == "shrinking · many grads") | (b == "growing · many grads")])


rows = []
rows.append(["as planned: openings weights, growth > 0, ratio > 1.0", *crowded(Mj)])
for rc in (0.5, 1.5, 2.0):
    rows.append([f"ratio cutoff {rc}", *crowded(Mj, rcut=rc)])
rows.append(["growth cutoff = NYC all-jobs 2022-32 (+12.05%)", *crowded(Mj, gcut=12.05)])
Je = job_table(B, wcol="weight_equal"); Me = major_table(B, Je, wcol="weight_equal")
rows.append(["equal weights (D2 range, other end)", *crowded(Me)])
# D20 off: D16 A as built, dispersed majors counted as supply (the earlier headline)
Jall = job_table(B_all); Mall = major_table(B_all, Jall)
rows.append(["D20 off: dispersed majors counted as supply (D16 A as built)", *crowded(Mall)])
# D16 B on the full bridge: named jobs first (kept for the record; D20 replaced it)
Bb = B_all.copy()
has_named = Bb[~Bb.is_all_other].major_award_key.unique()
Bb = Bb[~(Bb.is_all_other & Bb.major_award_key.isin(has_named))].copy()
Bb["weight_openings"] = Bb.weight_openings / Bb.groupby("major_award_key").weight_openings.transform("sum")
Jb = job_table(Bb); Mb = major_table(Bb, Jb)
rows.append(["D16 option B on the full bridge: All Other only when no named job", *crowded(Mb)])
Bc = B_all[~B_all.is_all_other].copy()
Bc["weight_openings"] = Bc.weight_openings / Bc.groupby("major_award_key").weight_openings.transform("sum")
Jc = job_table(Bc); Mc = major_table(Bc, Jc)
rows.append(["D16 option C on the full bridge: All Other links dropped", *crowded(Mc)])
avg = aw[aw.year.between(2019, 2024)].groupby(["cip_code", "award_level"]).awards_total.sum().div(6).rename("awards").reset_index()
Ba = bridge.merge(avg, on=["cip_code", "award_level"], how="left"); Ba["awards"] = Ba.awards.fillna(0)
Ja = job_table(Ba); Ma = major_table(Ba, Ja)
rows.append(["2019–2024 average awards instead of 2024", *crowded(Ma)])
cu = aw[(aw.year == 2024) & aw.is_cuny].groupby(["cip_code", "award_level"]).awards_total.sum().rename("awards").reset_index()
Bu = bridge.merge(cu, on=["cip_code", "award_level"], how="left"); Bu["awards"] = Bu.awards.fillna(0)
Ju = job_table(Bu); Mu = major_table(Bu, Ju)
rows.append(["CUNY awards only (ratio vs all NYC openings)", *crowded(Mu)])
# Added after the first run (not pre-registered, declared here): growth measured as actual NYC change 2019 → 2025 (D8 estimate, nb 40)
# instead of the 2022-32 projection, because 85% of linked jobs are projected to grow, which empties the "shrinking" boxes.
act = q("""
select soc_code, max(employment_est) filter (where year = 2019) e19, max(employment_est) filter (where year = 2025) e25
from fct_job_employment_nyc_est group by 1
""").set_index("soc_code")
act = ((act.e25 / act.e19 - 1) * 100).rename("actual_growth")
Jx = J.copy(); Jx["pct_change"] = Jx.index.map(act)
Mx = major_table(B, Jx)
rows.append(["growth = actual NYC 2019 → 2025 (D8 estimate) instead of projected (added after first run)", *crowded(Mx)])
sens = pd.DataFrame(rows, columns=["variant", "majors in 'more competition'", "their 2024 awards", "majors with ratio > cutoff"])
show(sens.round(0), "Sensitivity of the major grid")
# D20 in numbers: what the dispersed majors did to the job ratios while they were counted
d20 = pd.DataFrame({"supply D20 off": Jall.supply, "supply now": J.supply.reindex(Jall.index).fillna(0),
                    "openings": Jall.annual_openings_avg, "ratio D20 off": Jall.ratio, "ratio now": J.ratio.reindex(Jall.index)})
d20["removed"] = d20["supply D20 off"] - d20["supply now"]
d20 = d20.join(jobs.set_index("soc_code").soc_title)
show(d20.sort_values("removed", ascending=False).head(12)[["soc_title", "openings", "supply D20 off", "supply now", "removed", "ratio D20 off", "ratio now"]].round(1),
     "Jobs that lose the most counted supply under D20 (dispersed majors no longer counted)")
show(Mx[(box(Mx.growth, Mx.ratio) == "shrinking · many grads") & (Mx.awards >= 100)].sort_values("awards", ascending=False)
       .assign(major=lambda d: d.cip_code.map(maj.major), level=lambda d: d.award_level.map(LEVEL))[["level", "major", "awards", "ratio", "growth", "top job"]]
       .assign(**{"top job": lambda d: d["top job"].map(jobs.set_index("soc_code").soc_title)}).round(2),
     "Under actual 2019 → 2025 growth: majors with 100+ awards in 'more competition'")

**Sensitivity of the major grid**

,variant,majors in 'more competition',their 2024 awards,majors with ratio > cutoff
0,"as planned: openings weights, growth > 0, rati...",8,159.00,477
1,ratio cutoff 0.5,13,543.00,833
2,ratio cutoff 1.5,0,0.00,388
3,ratio cutoff 2.0,0,0.00,270
4,growth cutoff = NYC all-jobs 2022-32 (+12.05%),181,"4,932.00",477
5,"equal weights (D2 range, other end)",10,277.00,481
6,D20 off: dispersed majors counted as supply (D...,8,159.00,516
7,D16 option B on the full bridge: All Other onl...,8,159.00,447
8,D16 option C on the full bridge: All Other lin...,8,159.00,376
9,2019–2024 average awards instead of 2024,8,171.00,525


**Jobs that lose the most counted supply under D20 (dispersed majors no longer counted)**

,soc_title,openings,supply D20 off,supply now,removed,ratio D20 off,ratio now
soc_code,,,,,,,
11-9199,"Managers, All Other",858.80,"8,534.30","2,693.50","5,840.80",9.90,3.10
11-1021,General and Operations Managers,"9,781.00","2,299.00",893.50,"1,405.50",0.20,0.10
19-4061,Social Science Research Assistants,291.00,"1,549.80",406.70,"1,143.10",5.30,1.40
25-2031,"Secondary School Teachers, Except Special and ...","2,627.00","5,392.60","4,506.80",885.80,2.10,1.70
13-1111,Management Analysts,"4,444.00",767.20,119.90,647.30,0.20,0.00
33-1012,First-Line Supervisors of Police and Detectives,538.00,636.10,4.50,631.60,1.20,0.00
13-1082,Project Management Specialists,"2,917.80",251.30,14.00,237.30,0.10,0.00
13-1071,Human Resources Specialists,"3,099.00",216.70,19.60,197.10,0.10,0.00
11-2022,Sales Managers,"1,584.00",177.30,48.40,128.80,0.10,0.00


**Under actual 2019 → 2025 growth: majors with 100+ awards in 'more competition'**

,level,major,awards,ratio,growth,top job
major_award_key,,,,,,
23.0101|5,bachelor's,"English Language and Literature, General",837.00,1.65,-4.68,"Secondary School Teachers, Except Special and ..."
45.0601|5,bachelor's,"Economics, General",655.00,2.00,-1.31,"Secondary School Teachers, Except Special and ..."
50.0702|5,bachelor's,"Fine/Studio Arts, General",586.00,2.03,-32.40,"Fine Artists, Including Painters, Sculptors, a..."
50.0410|5,bachelor's,Illustration,535.00,2.15,-50.40,Set and Exhibit Designers
11.0701|3,associate's,Computer Science,422.00,1.28,-49.03,Computer Network Support Specialists
10.0304|5,bachelor's,"Animation, Interactive Technology, Video Graph...",178.00,1.17,-39.13,Special Effects Artists and Animators
31.0505|5,bachelor's,Exercise Science and Kinesiology,136.00,5.44,-46.35,Exercise Physiologists
13.1318|5,bachelor's,Social Studies Teacher Education,105.00,1.17,-3.58,"Secondary School Teachers, Except Special and ..."


In [9]:
stable = pd.DataFrame({"planned": box(Mj.growth, Mj.ratio), "equal": box(Me.growth.reindex(Mj.index), Me.ratio.reindex(Mj.index)),
                       "D16 B": box(Mb.growth.reindex(Mj.index), Mb.ratio.reindex(Mj.index)), "avg awards": box(Ma.growth.reindex(Mj.index), Ma.ratio.reindex(Mj.index))})
stable = stable.join(Mj[["level", "major", "awards"]])
same = (stable.planned == stable.equal) & (stable.planned == stable["D16 B"]) & (stable.planned == stable["avg awards"])
print(f"majors that keep the same box under all four weightings: {int(same.sum())} of {len(stable)} "
      f"({stable[same].awards.sum() / stable.awards.sum():.0%} of awards)")
show(stable[(stable.planned == "shrinking · many grads") & ~same].sort_values("awards", ascending=False).head(20),
     "Majors in 'more competition' as planned that move under another weighting")

majors that keep the same box under all four weightings: 1363 of 1614 (75% of awards)


**Majors in 'more competition' as planned that move under another weighting**

,planned,equal,D16 B,avg awards,level,major,awards
major_award_key,,,,,,,


## 5. Checks

In [10]:
chk = []
chk.append(["Σ supply over jobs vs Σ 2024 awards at linked majors × weight on jobs with openings",
            f"{J.supply.sum():,.0f} vs {(B.awards * B.weight_openings)[B.openings_basis != 'none'].sum():,.0f}"])
chk.append(["IPEDS 2024 first majors, NYC (step 4 tie-out 18,490 / 54,735)",
            f"{aw[(aw.year == 2024) & (aw.award_level == 3)].awards_total.sum():,.0f} / {aw[(aw.year == 2024) & (aw.award_level == 5)].awards_total.sum():,.0f}"])
chk.append(["CUNY 2024 (11,402 / 23,437)", f"{cuny24.xs(3, level=1).sum():,.0f} / {cuny24.xs(5, level=1).sum():,.0f}"])
for key, label in [("42.0101|5", "Psychology, General BA (dispersed: traced on the D20-off table)"), ("52.0301|5", "Accounting BA"), ("11.0701|5", "Computer Science BA"), ("51.3801|5", "Registered Nursing BS")]:
    Mt, Bt, Jt = (Mall, B_all, Jall) if key in DISPERSED else (Mj, B, J)
    if key in Mt.index:
        r = Mt.loc[key]
        links = Bt[Bt.major_award_key == key].sort_values("weight_openings", ascending=False)
        hand = (links.weight_openings / links.weight_openings[links.soc_code.isin(Jt[Jt.ratio.notna()].index)].sum()
                * links.soc_code.map(Jt.ratio)).sum()
        chk.append([f"Trace {label}: awards {r.awards:,.0f}, top job {r['top job']} ({r['top job weight']:.0%}), ratio {r.ratio:.2f}, growth {r.growth:+.1f}%",
                    f"hand recompute ratio {hand:.2f} ✓" if abs(hand - r.ratio) < 1e-6 else f"hand {hand:.2f} ✗"])
    else:
        chk.append([f"Trace {label}", "key not found"])
show(pd.DataFrame(chk, columns=["check", "result"]), "Checks")
acct = pd.DataFrame([["fct_degree_awards (first majors, levels 3/5)", len(aw)], ["bridge_major_job", len(bridge)],
                     ["jobs receiving graduates", len(J)], ["jobs with a ratio", int(J.ratio.notna().sum())],
                     ["majors × level scored", len(Mj)]], columns=["table", "rows"])
show(acct, "Row accounting")

**Checks**

,check,result
0,Σ supply over jobs vs Σ 2024 awards at linked ...,"43,488 vs 43,488"
1,"IPEDS 2024 first majors, NYC (step 4 tie-out 1...","18,490 / 54,735"
2,"CUNY 2024 (11,402 / 23,437)","11,402 / 23,437"
3,"Trace Psychology, General BA (dispersed: trace...",hand recompute ratio 8.77 ✓
4,"Trace Accounting BA: awards 1,630, top job Acc...",hand recompute ratio 0.19 ✓
5,"Trace Computer Science BA: awards 1,465, top j...",hand recompute ratio 0.53 ✓
6,"Trace Registered Nursing BS: awards 2,197, top...",hand recompute ratio 0.52 ✓


**Row accounting**

,table,rows
0,"fct_degree_awards (first majors, levels 3/5)",23214
1,bridge_major_job,3802
2,jobs receiving graduates,427
3,jobs with a ratio,333
4,majors × level scored,1614


## 6. Findings (from the tables above)

1. Supply is falling, and fastest at CUNY. NYC first-major awards 2019 → 2024: associate's 27,593 → 18,490 (−33%; CUNY −40%, 18,880 → 11,402), bachelor's 57,782 → 54,735 (−5%; CUNY −14%, other NYC colleges +3%). The associate's drop is concentrated in the transfer majors: Liberal Arts AA −60%, Business Administration AA −48%, Criminal Justice AA −73%, Physical Sciences AA −60%. Bachelor's: Accounting −36%, Business Administration −20%, Finance −26%, English −33%; Computer Science +55%, Econometrics +35%, Nursing +3%. (IPEDS 2024 is provisional; CUNY Programs 2025 in `fct_cuny_program` is the next check.)
2. Seven in ten bachelor's awards link to a job the ACS confirms; two in ten are "dispersed"; one in three associate's awards link at all. Bachelor's 2024: 37,380 of 54,735 awards (68%) sit in `linked` majors, 12,067 (22%) in the 73 `dispersed` majors (D20: Psychology 4,129, Business Administration 1,922, Criminal Justice 1,607, Political Science 1,276, Research and Experimental Psychology 853, Management Science 821), 3,713 `transfer_pathway`, 1,575 `not_in_crosswalk`; 11,869 of 18,490 associate's awards are `transfer_pathway` (D14), so the associate's grid covers 6,108 awards only. Supply counted on jobs = 43,488 awards.
3. The pre-registered "crowded" box is almost empty, with or without D20. Only 47 of 333 jobs with a ratio are projected to shrink 2022-32, so "shrinking · many graduates" holds 1 job (Mechanical Drafters, 67 graduates for 49 openings) and 8 majors with 159 awards, unchanged by D20. The ratio cutoff barely matters (1.5 → zero majors; 0.5 → 13 majors, 543 awards); the growth cutoff is everything (NYC average +12.05% → 181 majors, 4,932 awards; actual 2019 → 2025 growth → 113 majors, 4,574 awards). With ten-year projections that grow almost everywhere, S11's rule says NYC has few crowded majors, and it says so honestly.
4. The signal is in the other box: growing jobs that still receive more graduates than openings, and D20 halves it. 21 jobs with 7,389 openings a year receive 13,758 graduates (1.9 per opening; before D20: 22 jobs, 22,306 graduates, 2.8): Secondary School Teachers (4,507 for 2,627, from 88 majors), Managers, All Other (2,694 for 859, 3.1 per opening, from 92 majors; 8,534 and 9.9 before D20), Data Scientists (1,072 for 811), Natural Sciences Managers (632 for 96), Architectural and Engineering Managers (619 for 197), Social Science Research Assistants (407 for 291, 1.4; 5.3 before D20). The majors that land there are now the science, economics and arts bachelor's: Biology (1,715 awards, 1.9 graduates per opening, 84% routed to Secondary School Teachers), Econometrics (1,014; 2.1), Sociology (814; 2.7), English (837; 1.7), Economics (655; 2.0), Fine/Studio Arts (586; 2.0). 421 bachelor's majors with 14,136 awards (38% of linked bachelor's awards) sit in "growing · many" (460 and 22,824 before D20).
5. The big professional majors are under-supplied by NYC graduates. Registered Nursing 0.52 graduates per opening (3,830 for 7,410), Computer Science 0.53, Information Science 0.44, Accounting 0.19, Finance 0.10. NYC employers fill the rest from outside NYC and from experienced hires, which is exactly the S11 (5) caveat. (Business Administration 0.45 and Management Science 0.28 are now dispersed and have no ratio.)
6. **D20 in numbers.** The dispersed majors supplied 5,841 of Managers All Other's 8,534 weighted graduates (ratio 9.9 → 3.1), 1,406 of General and Operations Managers' 2,299, 1,143 of Social Science Research Assistants' 1,550 (5.3 → 1.4), 886 of Secondary School Teachers' 5,393 (2.1 → 1.7), 647 of Management Analysts' 767 and all 632 of Police Supervisors' 636 (1.2 → 0.0). Managers, All Other still tops the growing · many list at 3.1 from 92 majors, carried now by Sociology (75% routed there), Econometrics (41%) and the smaller social sciences. D16 B and C on the full bridge (kept in the sensitivity table) change the crowded count by nothing and the ratio > 1 count by −30 and −101 majors; D20 changes it by −39 (516 → 477).
7. **Robustness.** 1,363 of 1,614 major × level scores (75% of awards) keep the same box under openings weights, equal weights, D16 B and six-year average awards; no "more competition" major moves. Σ supply = Σ awards at linked majors to the unit (43,488); four majors recomputed by hand (Psychology on the D20-off table, 8.77); IPEDS and CUNY tie-outs hold.
8. **Gaps.** 94 jobs receive 943 graduates but have no published or estimated openings; 28 majors have no ratio at all. Six degree-reachable jobs with 5,600 openings a year (Substitute Teachers, Buyers, Clinical Lab Technologists, Special Education Teachers K-5, Sales Engineers, Property Appraisers) receive no NYC graduates because they are broad or aggregate codes the detailed-only bridge cannot reach: a bridge follow-up.
9. For the mart and the pages. Show the ratio as a range (openings vs equal weights) with the three caveats on every page; put the "growing · many" box in words ("more graduates than openings, even as the job grows") rather than folding it into "crowded"; keep S11's rule for the crowded label; dispersed majors get no ratio and show ACS destinations by job family and earnings (nb 44) with the line "graduates of this major work across many fields; no single job to count against"; show the supply trend (−33% associate's) on its own page, it is the strongest number in this notebook.

**Decisions taken:** D19, growth axis = projected 2022-32 with actual beside it (8 majors crowded); D20, dispersed status for the five convention fields, applied in this re-run.